In [4]:
!pip install pyspark

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Assignment") \
    .getOrCreate()

In [7]:
df = spark.read.csv(
    "Combined_dataset.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- product_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: integer (nullable = true)
 |-- initial_price: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- final_price: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- images: string (nullable = true)
 |-- delivery_options: string (nullable = true)
 |-- product_details: string (nullable = true)
 |-- breadcrumbs: string (nullable = true)
 |-- product_specifications: string (nullable = true)
 |-- amount_of_stars: string (nullable = true)
 |-- what_customers_said: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- sizes: string (nullable = true)
 |-- videos: string (nullable = true)
 |-- seller_information: string (nullable = true)
 |-- variations: string (nullable = true)
 |-- best_offer: string (nullable = true)
 |-- more_offers: string (null

Ans 1. Limitation of MapReduce:Traditional MapReduce writes intermediate results to disk after each operation, making it slower for iterative algorithms and interactive analytics. It has a complex programming model and lacks efficient real-time processing. Spark overcomes these limitations through in-memory computing and easier APIs.

Ans 2. Spark keeps intermediate data in RAM instead of repeatedly reading and writing to disk. Machine learning algorithms perform multiple iterations on the same data, so storing data in memory greatly improves processing speed.

Ans.3 dropDuplicates() removes records having the same product_id. But as there are no rows duplicate so no no row is removed

In [10]:
print("Rows Before:", df.count())
df_no_duplicates = df.dropDuplicates(["product_id"])
print("Rows After:", df_no_duplicates.count())
df_no_duplicates.show()

Rows Before: 1000
Rows After: 1000
+----------+--------------------+--------------------+------+-------------+-------------+--------+-------------+---------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|               title| product_description|rating|ratings_count|initial_price|discount|  final_price| currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers|            category|
+----------+--------------------+--------------------+------+-------------+----------

Ans 4. To filter and group data i will use category column
  
  First query:convert final_price from string to double and remove unnecessary symbols.
  
  second query:The query groups products by category and calculates the average final price of products in each category.

In [15]:
from pyspark.sql.functions import regexp_replace, col
df = df.withColumn(
    "final_price",
    regexp_replace(col("final_price"), "₹", "")
)

df = df.withColumn(
    "final_price",
    regexp_replace(col("final_price"), ",", "")
)

df = df.withColumn(
    "final_price",
    regexp_replace(col("final_price"), '"', "")
)
from pyspark.sql.types import DoubleType

df = df.withColumn(
    "final_price",
    col("final_price").cast(DoubleType())
)


In [18]:
from pyspark.sql.functions import avg, count

df.groupBy("category") \
  .agg(
      count("*").alias("row_count"),
      avg("final_price").alias("average_price")
  ) \
  .show()

+--------------------+---------+------------------+
|            category|row_count|     average_price|
+--------------------+---------+------------------+
|{""name"":""Coole...|        1|             799.0|
|{""name"":""Dress...|        1|             884.0|
|{""name"":""Ethni...|        4|             611.5|
|""url"":""https:/...|        1|             424.0|
| {""name"":""Girls""|        5|             562.4|
|""url"":""https:/...|        1|             999.0|
|""url"":""https:/...|       12|183.83333333333334|
|""size_and_fit"":...|        1|               4.0|
| {""name"":""Susie""|        1|             398.0|
|""size_and_fit"":...|        1|               2.0|
|        long sleeves|        1|               5.0|
|""url"":""https:/...|        1|             449.0|
|{""name"":""MOZAF...|        1|             799.0|
|""url"":""https:/...|        1|             595.0|
|""size_and_fit"":...|        1|               1.0|
|{""name"":""Unisex""|        1|               1.0|
|""url"":""h

Ans 5.  .na.drop(): removes rows containing null values.
        
        .na.fill():replaces null values with specified values.

In [21]:
# count number of null
from pyspark.sql.functions import col

df.filter(col("seller_name").isNull()).count()

1

In [46]:
#fill them
df_filled = df.na.fill(
    "Unknown",
    subset=["seller_name"]
)

df_filled.show(5)

+----------+--------------+--------------------+------+-------------+-------------+--------+-----------+---------+------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|         title| product_description|rating|ratings_count|initial_price|discount|final_price| currency|images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers|            category|
+----------+--------------+--------------------+------+-------------+-------------+--------+-----------+---------+------+--------------------+--------------------+-

Ans 6.Groups products by category and displays only categories containing more than 10 products.

In [23]:
from pyspark.sql.functions import count

df.groupBy("category") \
  .agg(count("*").alias("total_products")) \
  .filter("total_products > 10") \
  .show()

+--------------------+--------------+
|            category|total_products|
+--------------------+--------------+
|""url"":""https:/...|            12|
|""url"":""https:/...|            87|
|"[{""name"":""Clo...|            62|
|""url"":""https:/...|            76|
|""url"":""https:/...|            18|
|""size_and_fit"":...|            12|
|{""name"":""Sport...|            11|
| {""name"":""Women""|            79|
|{""name"":""Dress...|            25|
|   {""name"":""Men""|            37|
|""url"":""https:/...|            26|
|""url"":""https:/...|            23|
|  {""name"":""Tops""|            48|
|{""name"":""Tshir...|            17|
+--------------------+--------------+



Ans 7.Spark DataFrames are immutable. Every operation creates a new DataFrame and leaves the original DataFrame unchanged.

In [24]:
#to drop
new_df = df.drop("videos")

In [25]:
#Rename a column:
new_df = df.withColumnRenamed(
    "final_price",
    "selling_price"
)

Ans 8.I filtered products based on rating.
This query filters products having ratings greater than or equal to 5 and prices below 1000.

In [28]:
df.filter(
    (df.rating >= 5.0) &
    (df.final_price < 1000)
).show()

+----------+------------------+--------------------+------+-------------+-------------+--------+-----------+---------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|             title| product_description|rating|ratings_count|initial_price|discount|final_price| currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers|            category|
+----------+------------------+--------------------+------+-------------+-------------+--------+-----------+---------+----------

Ans 9.Null values can produce inaccurate calculations and should be handled before performing aggregation operations.

In [30]:
#Fill null prices:
df_clean = df.na.fill(
    0,
    subset=["final_price"]
)
#Calculate average:
from pyspark.sql.functions import avg

df_clean.agg(
    avg("final_price")
).show()

+----------------+
|avg(final_price)|
+----------------+
|         304.399|
+----------------+



Ans 10.
The query converts a timestamp column into TimestampType and renames it as event_time.

In [33]:
#My dataset has no timestamp column.

# So create one:

df = df.withColumn(
    "raw_timestamp",
    lit("2025-06-20 10:30:00")
)
#before conversion

df.select("product_id", "raw_timestamp").show(5, truncate=False)



+----------+-------------------+
|product_id|raw_timestamp      |
+----------+-------------------+
|8376765   |2025-06-20 10:30:00|
|9136281   |2025-06-20 10:30:00|
|17633752  |2025-06-20 10:30:00|
|1376949   |2025-06-20 10:30:00|
|13939916  |2025-06-20 10:30:00|
+----------+-------------------+
only showing top 5 rows


In [34]:
# after conversion
df = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

df.select("product_id", "event_time").show(5, truncate=False)

+----------+-------------------+
|product_id|event_time         |
+----------+-------------------+
|8376765   |2025-06-20 10:30:00|
|9136281   |2025-06-20 10:30:00|
|17633752  |2025-06-20 10:30:00|
|1376949   |2025-06-20 10:30:00|
|13939916  |2025-06-20 10:30:00|
+----------+-------------------+
only showing top 5 rows


Ans 11.Spark redistributes rows with the same category into the same partition before aggregation. This movement of data between partitions is called a shuffle and is considered a wide transformation.

In [35]:
df.groupBy("category").count()

DataFrame[category: string, count: bigint]

Ans 12.This query removes records where seller information is missing or the title is empty.

In [36]:
from pyspark.sql.functions import col

rows_before = df.count()

clean_df = df.filter(
    col("seller_name").isNotNull() &
    (col("title") != "")
)

rows_after = clean_df.count()
rows_removed = rows_before - rows_after

print("Rows Before Cleaning:", rows_before)
print("Rows After Cleaning :", rows_after)
print("Rows Removed        :", rows_removed)

clean_df.show(5)

Rows Before Cleaning: 1000
Rows After Cleaning : 999
Rows Removed        : 1
+----------+--------------+--------------------+------+-------------+-------------+--------+-----------+---------+------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|product_id|         title| product_description|rating|ratings_count|initial_price|discount|final_price| currency|images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers|            category|         event_time|
+----------+--------------+--------------------

Ans 13. The .agg() function computes multiple statistics in a single operation.

In [37]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("final_price").alias("minimum_price"),
    max("final_price").alias("maximum_price"),
    avg("final_price").alias("average_price")
).show()

+-------------+-------------+-------------+
|minimum_price|maximum_price|average_price|
+-------------+-------------+-------------+
|          1.0|        999.0|      304.399|
+-------------+-------------+-------------+



Ans 14.inferSchema=True automatically detects data types. If the dataset contains inconsistent values, Spark may infer incorrect data types or convert some values to null, leading to data quality issues.

In [41]:
df = spark.read.csv(
    "Combined_dataset.csv",
    header=True,
    inferSchema=True
)
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: integer (nullable = true)
 |-- initial_price: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- final_price: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- images: string (nullable = true)
 |-- delivery_options: string (nullable = true)
 |-- product_details: string (nullable = true)
 |-- breadcrumbs: string (nullable = true)
 |-- product_specifications: string (nullable = true)
 |-- amount_of_stars: string (nullable = true)
 |-- what_customers_said: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- sizes: string (nullable = true)
 |-- videos: string (nullable = true)
 |-- seller_information: string (nullable = true)
 |-- variations: string (nullable = true)
 |-- best_offer: string (nullable = true)
 |-- more_offers: string (null

Ans 15 This pipeline:
Removes duplicate products.
Replaces null prices with 0.
Groups products by category and calculates total revenue.

In [43]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn(
    "final_price",
    regexp_replace(col("final_price"), "[^0-9.]", "").cast("double")
)

In [44]:
# Remove duplicates
df1 = df.dropDuplicates(["product_id"])
# Fill null prices
df2 = df1.na.fill(
    0,
    subset=["final_price"]
)
#Calculate total revenue by category
from pyspark.sql.functions import sum

result = df2.groupBy("category") \
            .agg(
                sum("final_price")
                .alias("total_revenue")
            )

result.show()

+--------------------+-------------+
|            category|total_revenue|
+--------------------+-------------+
|{""name"":""Coole...|        799.0|
|{""name"":""Dress...|        884.0|
|{""name"":""Ethni...|       2446.0|
|""url"":""https:/...|        424.0|
| {""name"":""Girls""|       2812.0|
|""url"":""https:/...|       2206.0|
|""size_and_fit"":...|          4.0|
|""url"":""https:/...|        999.0|
|""size_and_fit"":...|          2.0|
|        long sleeves|          5.0|
| {""name"":""Susie""|        398.0|
|""url"":""https:/...|        449.0|
|{""name"":""MOZAF...|        799.0|
|""url"":""https:/...|        595.0|
|""size_and_fit"":...|          1.0|
|{""name"":""Unisex""|          1.0|
|""url"":""https:/...|       4241.0|
|{""name"":""Jockey""|        579.0|
|{""name"":""VASTR...|        974.0|
|""url"":""https:/...|          2.0|
+--------------------+-------------+
only showing top 20 rows
